# Week 15: Graph Neural Networks & Geometric Deep Learning
## Notebook 12: Equivariant GNNs: SE(3) & SO(3) Symmetry in Molecules

### 1. Conceptual Intuition & Theoretical Background
In 3D molecular structures and drug discovery, molecular properties must adhere to physical geometric laws:
- **Rotational Invariance ($SO(3)$):** Rotating a molecule in 3D space does not change its ground-state energy.
- **Translational Invariance:** Shifting a molecule by vector $\vec{t}$ does not alter its binding affinity.
- **SE(3) Equivariance:** If the molecule rotates or translates, vector outputs (such as atomic force vectors) must rotate or translate by the exact same transformation:
$$f(g \cdot x) = g \cdot f(x), \quad \forall g \in SE(3)$$

**EGNN (Equivariant Graph Neural Network, Satorras et al., 2021):**
Achieves exact $SE(3)$ equivariance without spherical harmonics using scalar coordinate updates:
1. **Relative Distance:** $d_{ij}^2 = \|x_i - x_j\|^2$ (Strictly Invariant).
2. **Message Computation:** $m_{ij} = \phi_m(h_i, h_j, d_{ij}^2)$.
3. **Coordinate Update (Equivariant):**
$$x_i^{(l+1)} = x_i^{(l)} + \sum_{j \neq i} (x_i^{(l)} - x_j^{(l)}) \cdot \phi_x(m_{ij})$$
4. **Node Feature Update (Invariant):**
$$h_i^{(l+1)} = \phi_h\left( h_i^{(l)}, \sum_{j \neq i} m_{ij} \right)$$

---

### 2. Scratch Python Implementation: EGNN Layer

In [ ]:
import numpy as np

class EGNNLayerScratch:
    def __init__(self, feat_dim=4):
        self.feat_dim = feat_dim
        # Linear layer weights for coordinate scalar scaling
        np.random.seed(42)
        self.w_coord = np.random.normal(0, 0.1, feat_dim)
        
    def forward(self, h, x):
        """
        h: Invariant node scalar features (N, feat_dim)
        x: Equivariant 3D spatial coordinates (N, 3)
        """
        N = len(x)
        x_next = x.copy()
        
        # All-pairs relative displacements and distances
        for i in range(N):
            coord_disp_sum = np.zeros(3)
            for j in range(N):
                if i == j: continue
                
                # Invariant distance
                diff_x = x[i] - x[j]
                d_sq = np.sum(diff_x ** 2)
                
                # Mock message scalar: dot product with scalar features
                msg_scalar = np.dot(h[i] + h[j], self.w_coord) * np.exp(-d_sq)
                
                # Equivariant update: linear combination of displacement vectors
                coord_disp_sum += diff_x * msg_scalar
                
            x_next[i] += coord_disp_sum
            
        return h, x_next

# Test Equivariance under 3D Rotation Matrix R
def get_random_rotation_matrix():
    q, _ = np.linalg.qr(np.random.randn(3, 3))
    if np.linalg.det(q) < 0:
        q[:, 0] *= -1
    return q

np.random.seed(42)
N = 4
h_feats = np.random.randn(N, 4)
x_coords = np.random.randn(N, 3)

egnn = EGNNLayerScratch(feat_dim=4)
R = get_random_rotation_matrix()

# Path 1: Forward pass, then rotate output coordinates
_, x_out1 = egnn.forward(h_feats, x_coords)
x_out1_rotated = np.dot(x_out1, R.T)

# Path 2: Rotate input coordinates, then execute forward pass
x_coords_rotated = np.dot(x_coords, R.T)
_, x_out2 = egnn.forward(h_feats, x_coords_rotated)

# Measure Equivariance Discrepancy: ||f(R x) - R f(x)||
equivariance_diff = np.max(np.abs(x_out1_rotated - x_out2))

print("--- EGNN SE(3) Equivariance Verification ---")
print(f"Max Absolute Equivariance Discrepancy: {equivariance_diff:.1e}")
print("Verified: Network output rotates identically with the input coordinate frame!")

### 3. Step-by-Step Code Walkthrough
1. **Invariant Distance Metric:** Computes $\|x_i - x_j\|^2$, which remains invariant under rotation and translation.
2. **Radial Vector Scaling:** Scales the directional vector $(x_i - x_j)$ by invariant scalars to preserve equivariance.
3. **Equivariance Verification:** Demonstrates mathematically that $\text{EGNN}(R x) = R \cdot \text{EGNN}(x)$.

---

### 4. Industry Applications & Trade-offs
- **Drug Discovery & AlphaFold:** Predicting atomic forces, protein folding dynamics, and ligand docking poses.
- **Trade-off:** Enforcing strict geometric equivariance limits the use of arbitrary non-linear feature combinations.

---

### 5. Top Interview Questions & Answers
**Q1: What is the difference between Invariance and Equivariance?**
> **Answer:** Invariance means the output does not change when the input transforms: $f(g \cdot x) = f(x)$ (e.g. molecular mass). Equivariance means the output transforms under the same group operation: $f(g \cdot x) = g \cdot f(x)$ (e.g. atomic forces).

**Q2: Why is SE(3) equivariance important in molecular generation?**
> **Answer:** Molecules in nature have no fixed coordinate axes. An equivariant model guarantees that rotating the input molecule produces a consistently rotated prediction, saving model capacity that would otherwise be spent learning orientation augmentations.